In [1]:
"""
Build a denormalized PR-level DataFrame from the AIDev-pop subset
(repos with >100 stars) for analyzing what practices correlate with
the quality of Agentic-PRs.

Requirements:
    pip install pandas pyarrow huggingface_hub
"""

import pandas as pd

BASE = "hf://datasets/hao-li/AIDev"

# ------------------------------------------------------------------ #
# 1. Load AIDev-pop tables                                            #
# ------------------------------------------------------------------ #
print("before reading first parquet")
pr           = pd.read_parquet(f"{BASE}/pull_request.parquet")
print("after reading pr")
repo         = pd.read_parquet(f"{BASE}/repository.parquet")
commits      = pd.read_parquet(f"{BASE}/pr_commits.parquet")
print("after pr_commits")
commit_files = pd.read_parquet(f"{BASE}/pr_commit_details.parquet")
print("after pr_commit_details")
reviews      = pd.read_parquet(f"{BASE}/pr_reviews.parquet")
review_cmts  = pd.read_parquet(f"{BASE}/pr_review_comments_v2.parquet")
issue_cmts   = pd.read_parquet(f"{BASE}/pr_comments.parquet")
task_type    = pd.read_parquet(f"{BASE}/pr_task_type.parquet")
rel_issue    = pd.read_parquet(f"{BASE}/related_issue.parquet")
print("all parquets downloaded")

# Parse timestamps (PR table)
for col in ["created_at", "closed_at", "merged_at"]:
    pr[col] = pd.to_datetime(pr[col], errors="coerce", utc=True)

# ------------------------------------------------------------------ #
# 2. Outcome / quality features (dependent variables)                 #
# ------------------------------------------------------------------ #
pr["is_merged"]          = pr["merged_at"].notna().astype(int)
pr["is_closed_unmerged"] = ((pr["state"] == "closed") & pr["merged_at"].isna()).astype(int)
pr["time_to_merge_hours"] = (pr["merged_at"] - pr["created_at"]).dt.total_seconds() / 3600
pr["time_to_close_hours"] = (pr["closed_at"] - pr["created_at"]).dt.total_seconds() / 3600

print("dependent variables set")

# ------------------------------------------------------------------ #
# 3. PR-size features (aggregate pr_commit_details to PR level)       #
#    NB: pr_commit_details has ONE ROW PER FILE PER COMMIT.           #
# ------------------------------------------------------------------ #
size_feats = (
    commit_files.groupby("pr_id")
    .agg(
        loc_added    = ("additions", "sum"),
        loc_deleted  = ("deletions", "sum"),
        loc_churn    = ("changes",   "sum"),
        files_changed= ("filename",  "nunique"),
    )
    .reset_index()
)

print("PR-size features created")

# ------------------------------------------------------------------ #
# 4. Commit-granularity features                                      #
# ------------------------------------------------------------------ #
# Per-commit size first (sum file-level changes within a sha)
per_commit_size = (
    commit_files.groupby(["pr_id", "sha"])
    .agg(commit_total=("changes", "sum"))
    .reset_index()
)
commit_size_stats = (
    per_commit_size.groupby("pr_id")
    .agg(
        mean_commit_size = ("commit_total", "mean"),
        std_commit_size  = ("commit_total", "std"),
        max_commit_size  = ("commit_total", "max"),
    )
    .reset_index()
)

# Commit counts + message length from pr_commits
granularity_feats = (
    commits.groupby("pr_id")
    .agg(
        n_commits           = ("sha",     "nunique"),
        mean_commit_msg_len = ("message", lambda s: s.fillna("").str.len().mean()),
    )
    .reset_index()
)

# ------------------------------------------------------------------ #
# 5. Review-effort features                                           #
# ------------------------------------------------------------------ #
n_reviews = reviews.groupby("pr_id").size().rename("n_reviews").reset_index()

n_changes_requested = (
    reviews[reviews["state"] == "CHANGES_REQUESTED"]
    .groupby("pr_id").size().rename("n_changes_requested").reset_index()
)
n_approvals = (
    reviews[reviews["state"] == "APPROVED"]
    .groupby("pr_id").size().rename("n_approvals").reset_index()
)

# Inline review comments: count via review -> pr_id (works whether or
# not pr_review_comments_v2 carries pr_id directly)
if "pr_id" in review_cmts.columns:
    n_inline_cmts = (
        review_cmts.groupby("pr_id").size()
        .rename("n_inline_review_comments").reset_index()
    )
else:
    n_inline_cmts = (
        review_cmts
        .merge(reviews[["id", "pr_id"]].rename(columns={"id": "pull_request_review_id"}),
               on="pull_request_review_id", how="left")
        .groupby("pr_id").size()
        .rename("n_inline_review_comments").reset_index()
    )

# Issue-thread comments from humans only (excludes bots, including the agent itself)
human_cmts = issue_cmts[issue_cmts["user_type"] == "User"]
n_issue_cmts = (
    human_cmts.groupby("pr_id").size()
    .rename("n_human_issue_comments").reset_index()
)

# ------------------------------------------------------------------ #
# 6. Task type and linked-issue indicator                             #
# ------------------------------------------------------------------ #
task_feats = task_type[["id", "type"]].rename(columns={"id": "pr_id", "type": "task_type"})

has_issue = (
    rel_issue.groupby("pr_id").size()
    .rename("n_linked_issues").reset_index()
)

# ------------------------------------------------------------------ #
# 7. PR-description features                                          #
# ------------------------------------------------------------------ #
pr["title_len"] = pr["title"].fillna("").str.len()
pr["body_len"]  = pr["body"].fillna("").str.len()

print("PR-description features set")

# ------------------------------------------------------------------ #
# 8. Build the master denormalized DataFrame                          #
# ------------------------------------------------------------------ #
df = pr.rename(columns={"id": "pr_id"}).copy()

# Attach repo metadata (controls / stratifiers)
df = df.merge(
    repo.rename(columns={
        "id": "repo_id",
        "language": "repo_language",
        "stars":    "repo_stars",
        "forks":    "repo_forks",
        "license":  "repo_license",
    })[["repo_id", "repo_language", "repo_stars", "repo_forks", "repo_license"]],
    on="repo_id", how="left",
)

# Left-join every PR-level aggregate
for table in [
    size_feats, granularity_feats, commit_size_stats,
    n_reviews, n_changes_requested, n_approvals,
    n_inline_cmts, n_issue_cmts,
    task_feats, has_issue,
]:
    df = df.merge(table, on="pr_id", how="left")

print("Left-join performed")

# Count features: NaN really means "zero of these happened"
count_cols = [
    "n_reviews", "n_changes_requested", "n_approvals",
    "n_inline_review_comments", "n_human_issue_comments",
    "n_linked_issues", "n_commits", "files_changed",
    "loc_added", "loc_deleted", "loc_churn",
]
df[count_cols] = df[count_cols].fillna(0).astype("int64")
df["has_linked_issue"] = (df["n_linked_issues"] > 0).astype(int)

print("features counted")

# ------------------------------------------------------------------ #
# 9. Sanity check + save                                              #
# ------------------------------------------------------------------ #
assert df["pr_id"].is_unique, "Row explosion in joins — investigate!"
print(f"Final shape           : {df.shape}")
print(f"PRs in pull_request   : {len(pr)}")
print(f"Merge rate            : {df['is_merged'].mean():.2%}")
print(f"Agents                : {df['agent'].value_counts().to_dict()}")
print(df.head())

df.to_parquet("aidev_pop_pr_features.parquet", index=False)
print("\nSaved -> aidev_pop_pr_features.parquet")

before reading first parquet
after reading pr
after pr_commits
after pr_commit_details
all parquets downloaded
dependent variables set
PR-size features created
PR-description features set
Left-join performed
features counted
Final shape           : (33596, 41)
PRs in pull_request   : 33596
Merge rate            : 71.48%
Agents                : {'OpenAI_Codex': 21799, 'Copilot': 4970, 'Devin': 4827, 'Cursor': 1541, 'Claude_Code': 459}
        pr_id  number                                              title  \
0  3264933329    2911  Fix: Wait for all partitions in load_collectio...   
1  3265118634       2                   ファイルパス参照を相対パスに統一し、doc/からdocs/に統一   
2  3265640341      30        Add build staleness detection for debug CLI   
3  3265709660     205  feat: add comprehensive README screenshots wit...   
4  3265782173   17625        chore: remove HashedPostStateProvider trait   

                                                body        agent    user_id  \
0  ## Summary\n\nFixes an

In [ ]:
"""
Analysis pipeline for AIDev-pop:
    What practices correlate with the quality of Agentic-PRs?

Practices (independent vars):
    PR size            : loc_added, loc_deleted, loc_changed, files_changed
    Commit granularity : n_commits, mean_commit_size, max_commit_size
    Task type          : pr_task_type.type  (Conventional Commit category)
    Description        : title_len, body_len
    Linked issue       : has_linked_issue

Quality outcomes (dependent vars):
    is_merged                  -> logistic regression
    time_to_merge_hours        -> Cox proportional hazards (with censoring)
    n_inline_review_comments   -> negative binomial regression
    n_changes_requested        -> negative binomial regression

Controls:
    agent, log(repo_stars), repo_language (top-10 + "other")

Standard errors clustered at the repository level throughout.

Requirements:
    pip install pandas numpy scipy statsmodels lifelines matplotlib seaborn
"""

import os
import warnings
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
from lifelines import CoxPHFitter
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

INPUT   = "aidev_pop_pr_features.parquet"
OUT_DIR = "analysis_outputs"
FIG_DIR = os.path.join(OUT_DIR, "figs")
os.makedirs(FIG_DIR, exist_ok=True)


# =================================================================== #
# 1. Load + prep                                                       #
# =================================================================== #
def load_and_prep(path):
    df = pd.read_parquet(path)

    # Ensure timestamps are tz-aware datetimes
    for c in ["created_at", "closed_at", "merged_at"]:
        df[c] = pd.to_datetime(df[c], errors="coerce", utc=True)

    # Drop rows missing the essentials
    df = df.dropna(subset=["created_at", "is_merged", "repo_id"]).copy()

    # Log-transform skewed predictors (all >= 0; log1p handles zeros)
    for c in ["loc_added", "loc_deleted", "loc_changed", "files_changed",
              "n_commits", "mean_commit_size", "max_commit_size",
              "title_len", "body_len", "repo_stars"]:
        df[f"log_{c}"] = np.log1p(df[c].fillna(0))

    # Top-K language grouping
    top_langs = df["repo_language"].value_counts().head(10).index
    df["repo_language_grp"] = (
        df["repo_language"].where(df["repo_language"].isin(top_langs), other="other")
                            .fillna("other")
    )

    # Group rare task types (<100 PRs) into "other"
    keep = df["task_type"].value_counts()
    keep = keep[keep >= 100].index
    df["task_type_grp"] = (
        df["task_type"].where(df["task_type"].isin(keep), other="other")
                       .fillna("missing")
    )

    # Cox duration: merged -> time-to-merge; closed-unmerged -> time-to-close;
    # still open -> time since creation (censored)
    now = pd.Timestamp.now(tz="UTC")
    df["duration_hours"] = np.where(
        df["is_merged"] == 1,
        df["time_to_merge_hours"],
        np.where(
            df["closed_at"].notna(),
            df["time_to_close_hours"],
            (now - df["created_at"]).dt.total_seconds() / 3600,
        ),
    )
    df["event"] = df["is_merged"].astype(int)

    return df


# =================================================================== #
# 2. Descriptive stats                                                 #
# =================================================================== #
def summary_stats(df):
    print("=" * 72); print("OVERALL"); print("=" * 72)
    print(f"PRs           : {len(df):,}")
    print(f"Repos         : {df['repo_id'].nunique():,}")
    print(f"Merge rate    : {df['is_merged'].mean():.2%}")
    print(f"Median LOC ch.: {df['loc_changed'].median():.0f}")
    print(f"Median commits: {df['n_commits'].median():.0f}\n")

    for col in ["agent", "task_type_grp"]:
        print("=" * 72); print(f"BY {col.upper()}"); print("=" * 72)
        agg = (df.groupby(col)
                 .agg(n=("pr_id", "size"),
                      merge_rate=("is_merged", "mean"),
                      median_loc_changed=("loc_changed", "median"),
                      median_commits=("n_commits", "median"),
                      median_review_cmts=("n_inline_review_comments", "median"))
                 .round(3)
                 .sort_values("n", ascending=False))
        print(agg); print()
        agg.to_csv(os.path.join(OUT_DIR, f"summary_by_{col}.csv"))


# =================================================================== #
# 3. Bivariate analysis                                                #
# =================================================================== #
def bivariate_continuous(df, practices, outcomes):
    rows = []
    for p in practices:
        for o in outcomes:
            sub = df[[p, o]].dropna()
            if len(sub) < 30:
                continue
            rho, pval = stats.spearmanr(sub[p], sub[o])
            rows.append({"practice": p, "outcome": o, "n": len(sub),
                         "spearman_rho": rho, "p_value": pval})
    return (pd.DataFrame(rows)
              .assign(abs_rho=lambda d: d["spearman_rho"].abs())
              .sort_values(["outcome", "abs_rho"], ascending=[True, False])
              .drop(columns="abs_rho"))


def kruskal_by_group(df, cat_col, num_col):
    groups = [g[num_col].dropna().values for _, g in df.groupby(cat_col)
              if len(g) >= 20]
    if len(groups) < 2:
        return np.nan, np.nan
    h, p = stats.kruskal(*groups)
    return h, p


def chi2_categorical_vs_binary(df, cat_col, outcome="is_merged"):
    ct = pd.crosstab(df[cat_col], df[outcome])
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    rates = (df.groupby(cat_col)[outcome]
               .agg(merge_rate="mean", n="size")
               .sort_values("merge_rate", ascending=False))
    return chi2, p, dof, rates


# =================================================================== #
# 4. Multivariate models                                               #
# =================================================================== #
RHS = (
    "log_loc_added + log_loc_deleted + log_files_changed "
    "+ log_n_commits + log_mean_commit_size "
    "+ log_title_len + log_body_len + has_linked_issue "
    "+ C(task_type_grp) + C(agent) "
    "+ log_repo_stars + C(repo_language_grp)"
)


def fit_logistic_merge(df):
    return (smf.logit(f"is_merged ~ {RHS}", data=df)
               .fit(cov_type="cluster", cov_kwds={"groups": df["repo_id"]},
                    disp=False, maxiter=200))


def fit_negbin(df, outcome):
    sub = df.dropna(subset=[outcome])
    return (smf.negativebinomial(f"{outcome} ~ {RHS}", data=sub)
               .fit(cov_type="cluster", cov_kwds={"groups": sub["repo_id"]},
                    disp=False, maxiter=200))


def fit_cox_time_to_merge(df):
    cols = [
        "duration_hours", "event",
        "log_loc_added", "log_loc_deleted", "log_files_changed",
        "log_n_commits", "log_mean_commit_size",
        "log_title_len", "log_body_len", "has_linked_issue",
        "log_repo_stars", "task_type_grp", "agent", "repo_language_grp",
        "repo_id",
    ]
    cox_df = df[cols].dropna()
    cox_df = cox_df[cox_df["duration_hours"] > 0]
    cox_df = pd.get_dummies(
        cox_df,
        columns=["task_type_grp", "agent", "repo_language_grp"],
        drop_first=True,
    )
    cph = CoxPHFitter(penalizer=0.01)
    cph.fit(
        cox_df.drop(columns=["repo_id"]),
        duration_col="duration_hours",
        event_col="event",
        cluster_col=None,        # lifelines clustering: pass weights/strata if needed
        robust=True,
    )
    return cph


def tidy_summary(model, kind):
    """Return a clean coefficient table with effect sizes."""
    s = model.summary2().tables[1].copy()
    s.columns = [c.lower().replace(".", "").replace(" ", "_") for c in s.columns]
    if kind == "logit":
        s["odds_ratio"] = np.exp(s["coef"])
    elif kind == "negbin":
        s["irr"] = np.exp(s["coef"])      # incidence rate ratio
    return s


# =================================================================== #
# 5. Plots                                                             #
# =================================================================== #
def make_plots(df, m_logit):
    sns.set_theme(style="whitegrid")

    # Merge rate breakdowns
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    df.groupby("task_type_grp")["is_merged"].mean().sort_values().plot(
        kind="barh", ax=axes[0], title="Merge rate by task type")
    df.groupby("agent")["is_merged"].mean().sort_values().plot(
        kind="barh", ax=axes[1], title="Merge rate by agent")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "merge_rate_breakdown.png"), dpi=120)
    plt.close()

    # Size / granularity vs outcomes
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.boxplot(data=df, x="is_merged", y="log_loc_changed", ax=axes[0])
    axes[0].set_title("log(1 + lines changed) vs merged")
    sample = df.sample(min(5000, len(df)), random_state=0)
    sns.scatterplot(data=sample, x="log_loc_changed",
                    y="n_inline_review_comments", alpha=0.3, ax=axes[1])
    axes[1].set_title("PR size vs inline review comments")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "size_vs_outcomes.png"), dpi=120)
    plt.close()

    # Forest plot of odds ratios from logistic model
    s = tidy_summary(m_logit, "logit")
    s = s[~s.index.str.contains("Intercept|repo_language|C\\(agent\\)|C\\(task_type")]
    s = s.sort_values("odds_ratio")
    fig, ax = plt.subplots(figsize=(9, max(4, 0.3 * len(s))))
    ax.errorbar(
        s["odds_ratio"], range(len(s)),
        xerr=[s["odds_ratio"] - np.exp(s["[0025"]),
              np.exp(s["0975]"]) - s["odds_ratio"]],
        fmt="o", color="black"
    )
    ax.axvline(1, ls="--", color="red", alpha=0.5)
    ax.set_yticks(range(len(s))); ax.set_yticklabels(s.index)
    ax.set_xscale("log"); ax.set_xlabel("Odds ratio (95% CI)")
    ax.set_title("Logistic: P(merged) — practice variables only")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "logit_forest.png"), dpi=120)
    plt.close()

# Wpływ różnych praktyk na jakość PRów tworzonych przez agentów

## Praktyki
- ### Rozmiar PRa
    - #### liczba zmienionych plików
    - #### liczba zmienionych linii (oraz dodanych i usuniętych)
- ### Granularność commitów
    - #### liczba commitów (w danym PRze)
    - #### rozmiar commita (ilość zmian, średnia i maksymalna)
- ### Typ PRa
    - #### feat, fix, doc itp

## Wyznaczniki jakości
- ### czy PR został zmergowany
- ### czas do zmergowania (W godzinach)
- ### ilość komentarzy (inline, czyli do konkretnej zmiany oraz ogólnie pod issue)
- ### ilość żądanych zmian

## Ładowanie danych

In [10]:
df = load_and_prep(INPUT)
print("load_and_prep DONE...")

load_and_prep DONE...


## Podsumowanie danych

In [11]:
summary_stats(df)
print("=" * 72)
print("SUMMARY_STATS DONE\nnow proceeding to calculate correlations")
print("=" * 72)

OVERALL
PRs           : 33,596
Repos         : 2,807
Merge rate    : 71.48%
Median LOC ch.: 95
Median commits: 1

BY AGENT
                  n  merge_rate  median_loc_changed  median_commits  \
agent                                                                 
OpenAI_Codex  21799       0.826                74.0             1.0   
Copilot        4970       0.430               179.0             3.0   
Devin          4827       0.538               166.0             3.0   
Cursor         1541       0.652               271.0             2.0   
Claude_Code     459       0.590               687.0             3.0   

              median_review_cmts  
agent                             
OpenAI_Codex                 0.0  
Copilot                      0.0  
Devin                        0.0  
Cursor                       0.0  
Claude_Code                  0.0  

BY TASK_TYPE_GRP
                   n  merge_rate  median_loc_changed  median_commits  \
task_type_grp                               

## Korelacje między praktykami a wyznacznikami jakości
(między zmiennymi liczbowymi)

Ogólna zasada dla współczynnika ρ (rho) Spearmana:
|ρ| ≈ 0.0–0.1 → praktycznie brak związku; 0.1–0.3 → słaby; 0.3–0.5 → umiarkowany; 0.5–0.7 → silny; 0.7+ → bardzo silny. Znak wskazuje kierunek.

In [12]:
practices = ["log_loc_added", "log_loc_deleted", "log_loc_changed",
                "log_files_changed", "log_n_commits", "log_mean_commit_size",
                "log_max_commit_size", "log_title_len", "log_body_len"]
outcomes  = ["is_merged", "time_to_merge_hours",
                "n_inline_review_comments", "n_changes_requested"]

print("=" * 72)
print("BIVARIATE: Spearman correlations (practice vs outcome)")
print("=" * 72)
biv = bivariate_continuous(df, practices, outcomes)
print(biv.to_string(index=False))
biv.to_csv(os.path.join(OUT_DIR, "bivariate_spearman.csv"), index=False)

BIVARIATE: Spearman correlations (practice vs outcome)
            practice                  outcome     n  spearman_rho       p_value
        log_body_len                is_merged 33596     -0.251257  0.000000e+00
       log_n_commits                is_merged 33596     -0.225817  0.000000e+00
       log_title_len                is_merged 33596     -0.210159  0.000000e+00
       log_loc_added                is_merged 33596     -0.139348 2.824099e-145
     log_loc_changed                is_merged 33596     -0.138559 1.212671e-143
 log_max_commit_size                is_merged 33596     -0.120199 2.481440e-108
     log_loc_deleted                is_merged 33596     -0.091650  1.385248e-63
   log_files_changed                is_merged 33596     -0.078990  1.203939e-47
log_mean_commit_size                is_merged 33596     -0.068204  6.153746e-36
       log_n_commits      n_changes_requested 33596      0.252731  0.000000e+00
        log_body_len      n_changes_requested 33596      0.172369

## liczba commitów w PRze (dość) silnie koreluje z czasem do zmergowania i ilością komentarzy do zmian
#### ale tylko słabo koreluje z tym, czy PR został zmergowany i ilością żądanych zmian
### ilość zmienionych linijek nie ma tak mocnych korelacji (a można by się ich spodziewać, szczególnie z ilością komentarzy, czy żądanych zmian)
### średni rozmiar commita także wykazuje tylko słabe korelacje z wyznacznikami jakości
## ciężko zatem mówić o wpływie granularności commitów na cokolwiek

## -
## -
## -
## -

## Korelacja między typem PRa a tym, czy został zmergowany
(zmienna kategoryczna vs binarna)

In [13]:
print("\n" + "=" * 72)
print("BIVARIATE: chi-square — task_type vs is_merged")
print("=" * 72)
chi2, p, dof, rates = chi2_categorical_vs_binary(df, "task_type_grp")
n = rates["n"].sum()
cramers_v = np.sqrt(chi2 / (n * min(rates.shape[0] - 1, 2 - 1)))
print(f"Cramér's V = {cramers_v:.3f}")
print(f"chi2={chi2:.1f}  dof={dof}  p={p:.2e}")
print(rates.round(3))
rates.to_csv(os.path.join(OUT_DIR, "merge_rate_by_task_type.csv"))


BIVARIATE: chi-square — task_type vs is_merged
Cramér's V = 0.137
chi2=629.6  dof=10  p=8.10e-129
               merge_rate      n
task_type_grp                   
docs                0.840   3887
ci                  0.791    411
test                0.784   2356
style               0.777    188
build               0.748    627
feat                0.711  14450
refactor            0.701   2288
chore               0.688    896
fix                 0.650   8106
perf                0.553    340
other               0.319     47


## p=8.10e-129 (niemal 0)
prawdopodobieństwo, że takie merge raty dla różnych typów PRów są przypadkowe jest niemal zerowe (zatem możemy statystycznie powiedzieć, że merge rate jest związany z typem PRa)
## Cramér's V = 0.137
mały rozmiar efektu

In [23]:
print("\n" + "=" * 72)
print("BIVARIATE: Kruskal-Wallis — task_type vs review effort")
print("=" * 72)
for o in ["n_inline_review_comments", "n_changes_requested",
            "time_to_merge_hours"]:
    h, pp = kruskal_by_group(df, "task_type_grp", o)
    print(f"  {o:<32}  H={h:.1f}  p={pp:.2e}")
    print(df.groupby("task_type_grp")[o].agg(mean="mean", n="size"))


BIVARIATE: Kruskal-Wallis — task_type vs review effort
  n_inline_review_comments          H=130.8  p=3.22e-23
                   mean      n
task_type_grp                 
build          0.754386    627
chore          0.819196    896
ci             0.564477    411
docs           0.864934   3887
feat           0.892318  14450
fix            0.726252   8106
other          0.191489     47
perf           0.744118    340
refactor       0.896416   2288
style          0.335106    188
test           0.386248   2356
  n_changes_requested               H=39.6  p=1.97e-05
                   mean      n
task_type_grp                 
build          0.035088    627
chore          0.026786    896
ci             0.043796    411
docs           0.039362   3887
feat           0.052734  14450
fix            0.055638   8106
other          0.000000     47
perf           0.035294    340
refactor       0.034965   2288
style          0.015957    188
test           0.033531   2356
  time_to_merge_hours      

In [15]:


    

    print("\n" + "=" * 72)
    print("LOGISTIC: P(merged) ~ practices + controls")
    print("=" * 72)
    m_logit = fit_logistic_merge(df)
    print(m_logit.summary())
    tidy_summary(m_logit, "logit").to_csv(os.path.join(OUT_DIR, "logit_merged.csv"))

    print("\n" + "=" * 72)
    print("NEG-BIN: n_inline_review_comments ~ practices + controls")
    print("=" * 72)
    m_nb1 = fit_negbin(df, "n_inline_review_comments")
    print(m_nb1.summary())
    tidy_summary(m_nb1, "negbin").to_csv(os.path.join(OUT_DIR, "negbin_inline_cmts.csv"))

    print("\n" + "=" * 72)
    print("NEG-BIN: n_changes_requested ~ practices + controls")
    print("=" * 72)
    m_nb2 = fit_negbin(df, "n_changes_requested")
    print(m_nb2.summary())
    tidy_summary(m_nb2, "negbin").to_csv(os.path.join(OUT_DIR, "negbin_changes_requested.csv"))

    print("\n" + "=" * 72)
    print("COX: time-to-merge ~ practices + controls")
    print("=" * 72)
    cph = fit_cox_time_to_merge(df)
    print(cph.summary)
    cph.summary.to_csv(os.path.join(OUT_DIR, "cox_time_to_merge.csv"))

    make_plots(df, m_logit)
    print(f"\nAll outputs saved under: {OUT_DIR}/")


if __name__ == "__main__":
    main()

IndentationError: unexpected indent (4183107616.py, line 1)

In [1]:
import pandas as pd

BASE = "hf://datasets/hao-li/AIDev"

In [3]:
pr           = pd.read_parquet(f"{BASE}/pull_request.parquet")
print(pr)
# commits      = pd.read_parquet(f"{BASE}/pr_commits.parquet")
# print("after pr_commits")

               id  number                                              title  \
0      3264933329    2911  Fix: Wait for all partitions in load_collectio...   
1      3265118634       2                   ファイルパス参照を相対パスに統一し、doc/からdocs/に統一   
2      3265640341      30        Add build staleness detection for debug CLI   
3      3265709660     205  feat: add comprehensive README screenshots wit...   
4      3265782173   17625        chore: remove HashedPostStateProvider trait   
...           ...     ...                                                ...   
33591  2857942945       2  fix: improve dark mode input focus and toggle ...   
33592  2857959763    2154  docs: add required packages to SeleniumScrapin...   
33593  2858280902     775           Fix heap buffer overflow in BundlePacket   
33594  2858429985     346         Improve environment variable documentation   
33595  2858527610     400          Add Doxygen Documentation for Public APIs   

                                       

In [2]:
commit_files = pd.read_parquet(f"{BASE}/pr_commit_details.parquet")
# print(commit_files)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
print(commit_files.head())

KeyboardInterrupt: 